# Tree-RL: Tree-Structured Reinforcement Learning for Sequential Object Localization
**Reproduce bài báo NIPS 2016** — Zequn Jie et al.

### Thứ tự chạy:
1. Cell 1: Cài packages
2. Cell 2: Download PASCAL VOC
3. Cell 3: Tất cả class/function definitions
4. Cell 4: Extract features (chạy 1 lần)
5. Cell 5: Training
6. Cell 6: Evaluation

In [ ]:
# ============================================================
# CELL 1: Cài đặt packages
# ============================================================
import sys
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128 -q
!{sys.executable} -m pip install opencv-python PyYAML tqdm scipy matplotlib tensorboard lxml -q
print('Done!')

In [ ]:
# ============================================================
# CELL 2: Download PASCAL VOC 2007 + 2012
# ============================================================
import os, sys, tarfile, urllib.request
from pathlib import Path

URLS = {
    'VOC2007_trainval': 'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar',
    'VOC2007_test':     'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar',
    'VOC2012_trainval': 'http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar',
}
DATA_DIR = Path('./VOCdevkit')
DATA_DIR.mkdir(exist_ok=True)
tar_dir = DATA_DIR / '_tars'
tar_dir.mkdir(exist_ok=True)

def download_progress(url, dest):
    fname = url.split('/')[-1]
    fpath = dest / fname
    if fpath.exists():
        print(f'  [Skip] {fname} already exists')
        return fpath
    print(f'  Downloading {fname}...')
    def hook(c, bs, ts):
        if ts > 0:
            sys.stdout.write(f'\r  {c*bs*100//ts}% ({c*bs/1e6:.0f}/{ts/1e6:.0f} MB)')
            sys.stdout.flush()
    urllib.request.urlretrieve(url, fpath, hook)
    print()
    return fpath

for name, url in URLS.items():
    year = '2007' if '2007' in name else '2012'
    voc_dir = DATA_DIR / f'VOC{year}'
    print(f'\n=== {name} ===')
    tar_path = download_progress(url, tar_dir)
    if not voc_dir.exists() or not any(voc_dir.iterdir()):
        print(f'  Extracting...')
        with tarfile.open(tar_path) as tar:
            tar.extractall(DATA_DIR)
        print(f'  Done -> {voc_dir}')
    else:
        print(f'  [Skip] {voc_dir} already extracted')

# Xác định đúng path (tránh lồng VOCdevkit/VOCdevkit)
if (DATA_DIR / 'VOCdevkit' / 'VOC2007').exists():
    VOC_ROOT = str(DATA_DIR / 'VOCdevkit')
else:
    VOC_ROOT = str(DATA_DIR)
print(f'\nVOC_ROOT = {VOC_ROOT}')

In [ ]:
# ============================================================
# CELL 3: Tất cả definitions (Actions, Metrics, Memory,
#          Dataset, FeatureExtractor, QNetwork, Agent)
# ============================================================
import random, copy, os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import deque, namedtuple
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.ops as ops
import torchvision.transforms as T
from torch.utils.data import Dataset
from PIL import Image
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── CONFIG ───────────────────────────────────────────────────
CFG = {
    'voc_root':          VOC_ROOT,
    'feature_cache_dir': './feature_cache',
    'checkpoint_dir':    './checkpoints',
    'log_dir':           './runs',
    # Model
    'roi_pool_size':     7,
    'hidden_dim':        1024,
    'num_actions':       13,
    'history_len':       50,
    # Actions
    'scaling_ratio':     0.55,
    'trans_ratio':       0.25,
    # Reward
    'iou_threshold':     0.5,
    'first_hit_bonus':   5.0,
    # Training
    'epochs':            25,
    'max_steps':         50,
    'batch_size':        64,
    'lr':                0.0001,
    'gamma':             0.9,
    'eps_start':         1.0,
    'eps_end':           0.1,
    'eps_decay_epochs':  10,
    'replay_capacity':   800_000,
    'replay_start':      10_000,
    'target_update_freq':1000,
    'save_every':        1,
    # State dim = 4096 + 4096 + 50*13
    'state_dim':         8842,
}
Path(CFG['checkpoint_dir']).mkdir(exist_ok=True)
Path(CFG['feature_cache_dir']).mkdir(exist_ok=True)

# ── ACTIONS ──────────────────────────────────────────────────
ACTION_NAMES = [
    'scale_top_left','scale_top_right','scale_bottom_left',
    'scale_bottom_right','scale_center',
    'trans_left','trans_right','trans_up','trans_down',
    'trans_wider','trans_narrower','trans_taller','trans_shorter',
]

def apply_action(window, action_id, sr=0.55, tr=0.25, img_w=1, img_h=1):
    x1,y1,x2,y2 = window
    w,h = x2-x1, y2-y1
    if action_id < 5:  # scaling
        sw,sh = w*sr, h*sr
        offsets = [(0,0),(w-sw,0),(0,h-sh),(w-sw,h-sh),((w-sw)/2,(h-sh)/2)]
        dx,dy = offsets[action_id]
        nx1,ny1,nx2,ny2 = x1+dx, y1+dy, x1+dx+sw, y1+dy+sh
    else:  # translation
        dx,dy = w*tr, h*tr
        moves = [(-dx,0),(dx,0),(0,-dy),(0,dy),(-dx,0,dx,0),(dx,0,-dx,0),(0,-dy,0,dy),(0,dy,0,-dy)]
        aid = action_id - 5
        if aid == 0:   nx1,ny1,nx2,ny2 = x1-dx,y1,x2-dx,y2
        elif aid == 1: nx1,ny1,nx2,ny2 = x1+dx,y1,x2+dx,y2
        elif aid == 2: nx1,ny1,nx2,ny2 = x1,y1-dy,x2,y2-dy
        elif aid == 3: nx1,ny1,nx2,ny2 = x1,y1+dy,x2,y2+dy
        elif aid == 4: nx1,ny1,nx2,ny2 = x1-dx,y1,x2+dx,y2
        elif aid == 5: nx1,ny1,nx2,ny2 = x1+dx,y1,x2-dx,y2
        elif aid == 6: nx1,ny1,nx2,ny2 = x1,y1-dy,x2,y2+dy
        else:          nx1,ny1,nx2,ny2 = x1,y1+dy,x2,y2-dy
    nx1 = max(0, min(nx1, img_w-2))
    ny1 = max(0, min(ny1, img_h-2))
    nx2 = max(nx1+1, min(nx2, img_w))
    ny2 = max(ny1+1, min(ny2, img_h))
    return np.array([nx1,ny1,nx2,ny2], dtype=np.float32)

def encode_history(history, hlen=50, nact=13):
    enc = np.zeros(hlen*nact, dtype=np.float32)
    for i,a in enumerate(history[-hlen:]):
        enc[i*nact+a] = 1.0
    return enc

# ── METRICS ──────────────────────────────────────────────────
def iou(a, b):
    ix1,iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2,iy2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    union = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union>0 else 0.0

def compute_reward(cur, nxt, gts, hit_flags, thr=0.5, bonus=5.0):
    new_flags = hit_flags.copy()
    for i,gt in enumerate(gts):
        if hit_flags[i]<1 and iou(nxt,gt)>=thr:
            new_flags[i]=1
            return bonus, new_flags
    max_sign = -1.0
    for gt in gts:
        if iou(nxt,gt)-iou(cur,gt)>0:
            max_sign=1.0; break
    return max_sign, new_flags

def recall_curve(proposals_list, gt_list, iou_thrs=[0.5,0.6,0.7,0.8], size_thr=2000):
    res = {'all':{},'large':{},'small':{}}
    for thr in iou_thrs:
        counts = {'all':[0,0],'large':[0,0],'small':[0,0]}
        for props,gts in zip(proposals_list,gt_list):
            for gt in gts:
                area = (gt[2]-gt[0])*(gt[3]-gt[1])
                sk = 'large' if area>size_thr else 'small'
                hit = int(any(iou(p,gt)>=thr for p in props))
                counts['all'][0]+=hit; counts['all'][1]+=1
                counts[sk][0]+=hit;   counts[sk][1]+=1
        for k in res:
            tot = counts[k][1]
            res[k][thr] = counts[k][0]/tot if tot>0 else 0.0
    return res

# ── REPLAY MEMORY (float16 để tiết kiệm RAM) ─────────────────
Transition = namedtuple('Transition',['state','action','reward','next_state','done'])

class ReplayMemory:
    def __init__(self, capacity=800_000):
        self.memory = deque(maxlen=capacity)
    def push(self, s, a, r, s_, done):
        self.memory.append(Transition(
            s.astype(np.float16), int(a), float(r),
            s_.astype(np.float16), bool(done)
        ))
    def sample(self, bs=64):
        batch = random.sample(self.memory, bs)
        return (
            np.stack([t.state      for t in batch]).astype(np.float32),
            np.array([t.action     for t in batch], dtype=np.int64),
            np.array([t.reward     for t in batch], dtype=np.float32),
            np.stack([t.next_state for t in batch]).astype(np.float32),
            np.array([t.done       for t in batch], dtype=np.float32),
        )
    def __len__(self): return len(self.memory)

# ── DATASET ──────────────────────────────────────────────────
VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle','bus','car','cat',
    'chair','cow','diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor'
]
CLS2IDX = {c:i for i,c in enumerate(VOC_CLASSES)}

class VOCDataset(Dataset):
    def __init__(self, root, years=['2007','2012'], split='trainval'):
        self.root=root; self.split=split
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        self.samples=[]
        for year in years:
            sf=os.path.join(root,f'VOC{year}','ImageSets','Main',f'{split}.txt')
            with open(sf) as f:
                self.samples+=[(year,l.strip()) for l in f if l.strip()]
        print(f'[VOCDataset] {split}: {len(self.samples)} images from {years}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        year,iid = self.samples[idx]
        img = Image.open(os.path.join(self.root,f'VOC{year}','JPEGImages',f'{iid}.jpg')).convert('RGB')
        W,H = img.size
        boxes,labels=[],[]
        ann = os.path.join(self.root,f'VOC{year}','Annotations',f'{iid}.xml')
        for obj in ET.parse(ann).getroot().findall('object'):
            diff=obj.find('difficult')
            if diff is not None and int(diff.text)==1: continue
            cls=obj.find('name').text.strip().lower()
            if cls not in CLS2IDX: continue
            bb=obj.find('bndbox')
            x1=max(0,float(bb.find('xmin').text)-1)
            y1=max(0,float(bb.find('ymin').text)-1)
            x2=min(W,float(bb.find('xmax').text)-1)
            y2=min(H,float(bb.find('ymax').text)-1)
            if x2>x1 and y2>y1:
                boxes.append([x1,y1,x2,y2]); labels.append(CLS2IDX[cls])
        gt = np.array(boxes,dtype=np.float32) if boxes else np.zeros((0,4),dtype=np.float32)
        return {'image':self.transform(img),'img_id':iid,'year':year,
                'img_w':W,'img_h':H,'gt_boxes':gt}

# ── FEATURE EXTRACTOR ────────────────────────────────────────
class VGG16Extractor(nn.Module):
    def __init__(self, roi_size=7):
        super().__init__()
        self.roi_size = roi_size
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.conv = nn.Sequential(*list(vgg.features.children())[:30])
        self.fc6  = nn.Sequential(
            nn.Linear(512*roi_size*roi_size, 4096),
            nn.ReLU(inplace=True), nn.Dropout(0.5)
        )
        if roi_size==7:
            self.fc6[0].weight.data = vgg.classifier[0].weight.data
            self.fc6[0].bias.data   = vgg.classifier[0].bias.data
        for p in self.conv.parameters(): p.requires_grad=False

    def get_fmap(self, img):
        with torch.no_grad(): return self.conv(img)

    def roi_feat(self, fmap, boxes, scale=1/16):
        bidx = torch.zeros(len(boxes),1,dtype=boxes.dtype,device=boxes.device)
        rois = torch.cat([bidx,boxes],1)
        pooled = ops.roi_pool(fmap,rois,(self.roi_size,self.roi_size),scale)
        return self.fc6(pooled.view(len(boxes),-1))

# ── Q-NETWORK ────────────────────────────────────────────────
class QNetwork(nn.Module):
    def __init__(self, state_dim=8842, hidden=1024, n_act=13):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim,hidden), nn.ReLU(),
            nn.Linear(hidden,hidden),   nn.ReLU(),
            nn.Linear(hidden,hidden),   nn.ReLU(),
            nn.Linear(hidden,n_act),
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias,0)
    def forward(self,x): return self.net(x)
    def two_best(self, s):
        with torch.no_grad():
            q = self.forward(s.unsqueeze(0)).squeeze(0)
            return int(q[:5].argmax()), int(q[5:].argmax())+5
    def greedy(self, s):
        with torch.no_grad():
            return int(self.forward(s.unsqueeze(0)).squeeze(0).argmax())

# ── AGENT ────────────────────────────────────────────────────
class TreeRLAgent:
    def __init__(self, extractor, q_net, target_net, cfg, device):
        self.ext=extractor; self.q=q_net; self.tgt=target_net
        self.dev=device; self.cfg=cfg

    def _build_state(self, gf, rf, hist):
        return np.concatenate([
            rf.cpu().numpy(), gf.cpu().numpy(),
            encode_history(hist, self.cfg['history_len'], self.cfg['num_actions'])
        ]).astype(np.float32)

    def _get_feats(self, fmap, box, W, H):
        scale = 1/16
        gb = torch.tensor([[0,0,W,H]],dtype=torch.float32,device=self.dev)
        with torch.no_grad():
            gf = self.ext.roi_feat(fmap,gb,scale).squeeze(0)
        wb = torch.from_numpy(np.array([box],dtype=np.float32)).to(self.dev)
        with torch.no_grad():
            rf = self.ext.roi_feat(fmap,wb,scale).squeeze(0)
        return gf, rf

    def run_episode(self, fmap, W, H, gt_boxes, eps):
        win = np.array([0,0,W,H],dtype=np.float32)
        hist=[]; hit_flags=np.full(len(gt_boxes),-1.0)
        gf,_ = self._get_feats(fmap,win,W,H)
        transitions=[]
        for step in range(self.cfg['max_steps']):
            _,rf = self._get_feats(fmap,win,W,H)
            state = self._build_state(gf,rf,hist)
            # eps-greedy (tree variant)
            if random.random()<eps:
                act = random.randint(0,12)
            else:
                st = torch.tensor(state,device=self.dev)
                bs,bt = self.q.two_best(st)
                act = random.choice([bs,bt])
            nwin = apply_action(win,act,
                self.cfg['scaling_ratio'],self.cfg['trans_ratio'],W,H)
            rew,hit_flags = compute_reward(
                win,nwin,gt_boxes,hit_flags,
                self.cfg['iou_threshold'],self.cfg['first_hit_bonus'])
            hist.append(act)
            _,nrf = self._get_feats(fmap,nwin,W,H)
            nstate = self._build_state(gf,nrf,hist)
            done = (step==self.cfg['max_steps']-1)
            transitions.append((state,act,rew,nstate,float(done)))
            win=nwin
        return transitions

    def tree_search(self, fmap, W, H, levels=5):
        scale=1/16
        gb=torch.tensor([[0,0,W,H]],dtype=torch.float32,device=self.dev)
        with torch.no_grad():
            gf=self.ext.roi_feat(fmap,gb,scale).squeeze(0)
        props=[]; queue=deque()
        queue.append((np.array([0,0,W,H],dtype=np.float32),[],1))
        while queue:
            win,hist,lvl=queue.popleft()
            props.append(win.copy())
            if lvl>=levels: continue
            wb=torch.from_numpy(np.array([win],dtype=np.float32)).to(self.dev)
            with torch.no_grad():
                rf=self.ext.roi_feat(fmap,wb,scale).squeeze(0)
            state=self._build_state(gf,rf,hist)
            st=torch.tensor(state,device=self.dev)
            bs,bt=self.q.two_best(st)
            for a in [bs,bt]:
                nw=apply_action(win,a,self.cfg['scaling_ratio'],
                                self.cfg['trans_ratio'],W,H)
                queue.append((nw,hist+[a],lvl+1))
        return props

print('All definitions loaded OK!')

In [ ]:
# ============================================================
# CELL 4: Pre-compute conv5_3 feature maps (chạy 1 lần)
# Nếu đã có cache rồi thì skip cell này
# ============================================================
extractor = VGG16Extractor(roi_size=CFG['roi_pool_size']).to(DEVICE)
extractor.eval()

transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def extract_split(root, year, split):
    out_dir = Path(CFG['feature_cache_dir']) / f'VOC{year}' / split
    out_dir.mkdir(parents=True, exist_ok=True)
    sf = os.path.join(root, f'VOC{year}', 'ImageSets', 'Main', f'{split}.txt')
    with open(sf) as f:
        ids = [l.strip() for l in f if l.strip()]
    skipped=0
    for iid in tqdm(ids, desc=f'VOC{year} {split}'):
        out = out_dir / f'{iid}.pt'
        if out.exists(): skipped+=1; continue
        img_path=os.path.join(root,f'VOC{year}','JPEGImages',f'{iid}.jpg')
        img=Image.open(img_path).convert('RGB')
        W,H=img.size
        img_t=transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            fmap=extractor.get_fmap(img_t).squeeze(0).half().cpu()
        torch.save({'feature_map':fmap,'img_w':W,'img_h':H}, out)
    print(f'Done: {len(ids)-skipped} extracted, {skipped} skipped')

extract_split(CFG['voc_root'], '2007', 'trainval')
extract_split(CFG['voc_root'], '2012', 'trainval')
extract_split(CFG['voc_root'], '2007', 'test')

In [ ]:
# ============================================================
# CELL 5: TRAINING
# ============================================================
import warnings; warnings.filterwarnings('ignore')

def load_fmap(year, split, iid):
    p = Path(CFG['feature_cache_dir'])/f'VOC{year}'/split/f'{iid}.pt'
    if not p.exists(): return None,None,None
    d=torch.load(p,map_location='cpu')
    return d['feature_map'].float().unsqueeze(0).to(DEVICE), d['img_w'], d['img_h']

def get_eps(epoch):
    s,e,d = CFG['eps_start'],CFG['eps_end'],CFG['eps_decay_epochs']
    return e if epoch>=d else s-(s-e)*(epoch/d)

def compute_loss(batch, q, tgt, gamma):
    S,A,R,S_,D = batch
    S  = torch.tensor(S,  dtype=torch.float32, device=DEVICE)
    A  = torch.tensor(A,  dtype=torch.int64,   device=DEVICE)
    R  = torch.tensor(R,  dtype=torch.float32, device=DEVICE)
    S_ = torch.tensor(S_, dtype=torch.float32, device=DEVICE)
    D  = torch.tensor(D,  dtype=torch.float32, device=DEVICE)
    qsa = q(S).gather(1,A.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        qnext = tgt(S_).max(1)[0]
        target = R + gamma*qnext*(1-D)
    return nn.HuberLoss()(qsa, target)

# Init models
extractor = VGG16Extractor(CFG['roi_pool_size']).to(DEVICE)
extractor.eval()

q_net     = QNetwork(CFG['state_dim'], CFG['hidden_dim'], CFG['num_actions']).to(DEVICE)
target_net= copy.deepcopy(q_net)
target_net.eval()
for p in target_net.parameters(): p.requires_grad=False

agent  = TreeRLAgent(extractor, q_net, target_net, CFG, DEVICE)
memory = ReplayMemory(CFG['replay_capacity'])
optimizer = optim.Adam(q_net.parameters(), lr=CFG['lr'])
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# Dataset
train_ds = VOCDataset(CFG['voc_root'], years=['2007','2012'], split='trainval')

# Resume nếu có checkpoint
start_epoch=0; global_step=0
ckpt_dir=Path(CFG['checkpoint_dir'])
existing = sorted(ckpt_dir.glob('epoch_*.pth'))
if existing:
    ckpt=torch.load(existing[-1], map_location=DEVICE)
    q_net.load_state_dict(ckpt['q_net'])
    target_net.load_state_dict(ckpt['target_net'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch=ckpt['epoch']+1
    global_step=ckpt.get('global_step',0)
    print(f'Resumed from {existing[-1].name} (epoch {start_epoch})')

print(f'Training Tree-RL | {len(train_ds)} images | {CFG["epochs"]} epochs')
print(f'Replay: {CFG["replay_capacity"]:,} | Batch: {CFG["batch_size"]} | GPU: {DEVICE}')
print('='*60)

for epoch in range(start_epoch, CFG['epochs']):
    eps = get_eps(epoch)
    indices = list(range(len(train_ds)))
    random.shuffle(indices)
    losses,rewards=[],[]
    q_net.train()

    pbar = tqdm(indices, desc=f'Epoch {epoch+1}/{CFG["epochs"]} ε={eps:.2f}')
    for idx in pbar:
        s=train_ds[idx]
        if len(s['gt_boxes'])==0: continue

        fmap,W,H = load_fmap(s['year'],'trainval',s['img_id'])
        if fmap is None:
            img_t=s['image'].unsqueeze(0).to(DEVICE)
            with torch.no_grad(): fmap=extractor.get_fmap(img_t)
            W,H=s['img_w'],s['img_h']

        trans=agent.run_episode(fmap,W,H,s['gt_boxes'],eps)
        ep_r=0
        for (st,a,r,st_,d) in trans:
            memory.push(st,a,r,st_,d); ep_r+=r
        rewards.append(ep_r)

        if len(memory)>=CFG['replay_start'] and len(memory)>=CFG['batch_size']:
            batch=memory.sample(CFG['batch_size'])
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                loss=compute_loss(batch,q_net,target_net,CFG['gamma'])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(q_net.parameters(),10.0)
            scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
            global_step+=1
            if global_step%CFG['target_update_freq']==0:
                target_net.load_state_dict(q_net.state_dict())

        if len(losses)>0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.4f}',
                             reward=f'{np.mean(rewards[-50:]):.1f}',
                             mem=len(memory))

    al=np.mean(losses) if losses else 0
    ar=np.mean(rewards) if rewards else 0
    print(f'Epoch {epoch+1}: loss={al:.4f}, reward={ar:.2f}, mem={len(memory):,}')

    ckpt_path = ckpt_dir/f'epoch_{epoch+1:03d}.pth'
    torch.save({'epoch':epoch,'global_step':global_step,
                'q_net':q_net.state_dict(),'target_net':target_net.state_dict(),
                'optimizer':optimizer.state_dict()}, ckpt_path)

print('Training complete!')

In [ ]:
# ============================================================
# CELL 6: EVALUATION — Recall table (Table 2 bài báo)
# ============================================================

# Load checkpoint tốt nhất
ckpt_dir = Path(CFG['checkpoint_dir'])
existing = sorted(ckpt_dir.glob('epoch_*.pth'))
if not existing:
    raise FileNotFoundError('Chưa có checkpoint! Chạy Cell 5 trước.')

ckpt = torch.load(existing[-1], map_location=DEVICE)
q_net.load_state_dict(ckpt['q_net'])
q_net.eval()
print(f'Loaded: {existing[-1].name}')

test_ds = VOCDataset(CFG['voc_root'], years=['2007'], split='test')

def run_eval(levels, max_images=None):
    all_props, all_gts = [], []
    n = len(test_ds) if max_images is None else min(max_images, len(test_ds))
    for i in tqdm(range(n), desc=f'Tree search levels={levels}'):
        s=test_ds[i]
        if len(s['gt_boxes'])==0: continue
        fmap,W,H=load_fmap(s['year'],'test',s['img_id'])
        if fmap is None:
            img_t=s['image'].unsqueeze(0).to(DEVICE)
            with torch.no_grad(): fmap=extractor.get_fmap(img_t)
            W,H=s['img_w'],s['img_h']
        with torch.no_grad():
            props=agent.tree_search(fmap,W,H,levels=levels)
        all_props.append(props)
        all_gts.append(s['gt_boxes'])

    res=recall_curve(all_props,all_gts)
    n_props=2**levels-1
    print(f'\n{"="*60}')
    print(f'Tree-RL Recall | Levels={levels}, Proposals={n_props}')
    print(f'{"="*60}')
    print(f'{"Category":<10} {"IoU=0.5":>9} {"IoU=0.6":>9} {"IoU=0.7":>9} {"IoU=0.8":>9}')
    print(f'{"-"*50}')

    # Paper reference values for levels=5 (31 proposals)
    paper = {'all':{0.5:68.1,0.6:58.7,0.7:43.8},
             'large':{0.5:78.9,0.6:69.8,0.7:53.3},
             'small':{0.5:23.2,0.6:12.5,0.7:4.5}}

    for k in ['all','large','small']:
        vals=[res[k].get(t,0)*100 for t in [0.5,0.6,0.7,0.8]]
        row=f'{k.capitalize():<10} '+' '.join(f'{v:>9.1f}' for v in vals)
        # Thêm so sánh với paper nếu levels=5
        if levels==5:
            refs=[paper[k].get(t,0) for t in [0.5,0.6,0.7]]
            diffs=[f'({v-r:+.1f})' for v,r in zip(vals[:3],refs)]
            row+='  vs paper: '+' '.join(diffs)
        print(row)
    print(f'{"="*60}')
    return res

# Chạy với 31 proposals (5 levels) và 63 proposals (6 levels)
res_31 = run_eval(levels=5)   # 31 proposals — Table 2 bài báo
res_63 = run_eval(levels=6)   # 63 proposals